# 00 — Validate the image dataset

## Objective

Check train, validation and test images for readability, minimum dimensions, colour modes and exact duplicates. Compare duplicate annotations before the existing final cleanup step.

### Running this review copy

Use the original project `notebooks` folder as the kernel working directory. The existing `../data` and `../models` paths depend on that working directory. Run the workflow in order in a fresh kernel. Stored outputs are retained from the source notebook and were not regenerated during this readability review. Review the final data-changing cleanup section before running Notebook 00.

## Setup and dataset locations

Import dependencies and define the three existing dataset folders, accepted image extensions and minimum dimensions. These checks do not create a new split.

In [11]:
from pathlib import Path
from PIL import Image
import hashlib
import pandas as pd

# ---------------------------------------------------------
# SET DATASET PATH
# ---------------------------------------------------------

# Define the three dataset directories.
DATASET_DIRS = {
    "train": Path("../data/raw/train"),
    "valid": Path("../data/raw/valid"),
    "test": Path("../data/raw/test")
}

VALID_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

MIN_WIDTH = 32
MIN_HEIGHT = 32

# Verify that each directory exists.
for split, folder in DATASET_DIRS.items():
    print(f"{split}: {folder.exists()} - {folder.resolve()}")

# Your later model divides images into 32x32 regions.
# Images smaller than this cannot be processed correctly.


train: True - C:\Users\bengr\OneDrive - Monash University\Desktop\Uni\3001 Numerical Methods and Machine Learning\Pork Rasher Inspection\data\raw\train
valid: True - C:\Users\bengr\OneDrive - Monash University\Desktop\Uni\3001 Numerical Methods and Machine Learning\Pork Rasher Inspection\data\raw\valid
test: True - C:\Users\bengr\OneDrive - Monash University\Desktop\Uni\3001 Numerical Methods and Machine Learning\Pork Rasher Inspection\data\raw\test


## Scan image files

Decode each image and record its dimensions, colour mode and SHA-256 file hash. Exact hashes identify byte-for-byte duplicates; they do not detect every visually similar image.

In [12]:

# ---------------------------------------------------------
# FUNCTION: CALCULATE IMAGE HASH
# ---------------------------------------------------------

def calculate_sha256(file_path):
    """
    Calculate a SHA-256 hash from the contents of an image file.

    Identical hashes indicate identical file contents, even
    when the filenames or folder locations are different.
    """

    sha256 = hashlib.sha256()

    with open(file_path, "rb") as file:
        # Read in chunks to avoid loading large files into memory.
        for chunk in iter(lambda: file.read(8192), b""):
            sha256.update(chunk)

    return sha256.hexdigest()


# ---------------------------------------------------------
# SCAN ALL THREE DATASET FOLDERS
# ---------------------------------------------------------

records = []

# Iterate through training, validation and testing folders.
for split, image_dir in DATASET_DIRS.items():

    print(f"Scanning {split} dataset...")

    # Search recursively for images.
    for file_path in image_dir.rglob("*"):

        if not file_path.is_file():
            continue

        extension = file_path.suffix.lower()

        # Ignore non-image files, including COCO JSON annotations.
        if extension not in VALID_EXTENSIONS:
            continue

        # Create a record for each image.
        record = {
            "split": split,
            "filename": file_path.name,
            "path": str(file_path),
            "extension": extension,
            "valid": True,
            "error": None,
            "width": None,
            "height": None,
            "mode": None,
            "sha256": None
        }

        try:
            # Verify that the image file is not corrupted.
            with Image.open(file_path) as img:
                img.verify()

            # Reopen the image to access pixel data and properties.
            with Image.open(file_path) as img:

                # Force pixel data to be decoded.
                img.load()

                record["width"] = img.width
                record["height"] = img.height
                record["mode"] = img.mode

                # Check minimum dimensions for 32x32 patch processing.
                if img.width < MIN_WIDTH or img.height < MIN_HEIGHT:
                    record["valid"] = False
                    record["error"] = "Image smaller than 32x32 pixels"

            # Calculate hash for duplicate detection.
            record["sha256"] = calculate_sha256(file_path)

        except Exception as e:

            # Record corrupted or unreadable images.
            record["valid"] = False
            record["error"] = str(e)

        records.append(record)


# Convert results into a DataFrame.
df_images = pd.DataFrame(records)

# Stop if no images were discovered.
if df_images.empty:
    raise ValueError("No images found. Check DATASET_DIRS.")

print("\nDATASET SCAN COMPLETE")
print("-" * 40)

print(f"Total images scanned: {len(df_images)}")

# Display the number of images in each dataset split.
display(df_images.groupby("split").size().rename("Image count"))

display(df_images.head())

Scanning train dataset...
Scanning valid dataset...
Scanning test dataset...

DATASET SCAN COMPLETE
----------------------------------------
Total images scanned: 3547


split
test       80
train    3347
valid     120
Name: Image count, dtype: int64

,split,filename,path,extension,valid,error,width,height,mode,sha256
0,train,frame_with_red_10049_jpg.rf.cd786c4f9dd783e72b...,..\data\raw\train\frame_with_red_10049_jpg.rf....,.jpg,True,None,720,540,RGB,95f494f56645fe3a65342bc14e9428b82a07918ad86d93...
1,train,frame_with_red_10049_jpg.rf.d742c71486ef35ee0d...,..\data\raw\train\frame_with_red_10049_jpg.rf....,.jpg,True,None,720,540,RGB,54190c9731509f42039bd9f7b532ddbd7a06b775ca6913...
2,train,frame_with_red_10049_jpg.rf.debe2aadb81b4c3985...,..\data\raw\train\frame_with_red_10049_jpg.rf....,.jpg,True,None,720,540,RGB,72e66e36fa752376e0aa3a3d1b6c8fc23931d1a5d73ffe...
3,train,frame_with_red_10050_jpg.rf.9c1d242a09e36de1f7...,..\data\raw\train\frame_with_red_10050_jpg.rf....,.jpg,True,None,720,540,RGB,4d8ef4b22bc1ba4ee3b41568ccb35e9bafe14d7c748cbc...
4,train,frame_with_red_10050_jpg.rf.d9a41729948289ba87...,..\data\raw\train\frame_with_red_10050_jpg.rf....,.jpg,True,None,720,540,RGB,e5be1919a4d01aaafb32d5f7025d61c4baf21be40bfd8b...


## Review invalid images

Inspect decoding failures and images smaller than the required 32 × 32 pixels.

In [13]:
# ---------------------------------------------------------
# INVALID IMAGE CHECK
# ---------------------------------------------------------

# Select all images marked as invalid during scanning.
invalid_images = df_images[
    ~df_images["valid"]
].copy()

print("INVALID IMAGE CHECK")
print("-" * 40)

print(f"Total invalid images: {len(invalid_images)}")

if not invalid_images.empty:

    # Display which split contains each invalid image.
    display(
        invalid_images[
            ["split", "filename", "path", "width", "height", "error"]
        ]
    )

else:
    print("No invalid images detected.")

INVALID IMAGE CHECK
----------------------------------------
Total invalid images: 0
No invalid images detected.


## Inspect dimensions and colour modes

Compare image sizes across splits and identify images requiring RGB conversion.

In [14]:
# ---------------------------------------------------------
# IMAGE DIMENSIONS AND COLOUR MODES
# ---------------------------------------------------------

print("IMAGE DIMENSIONS")
print("-" * 40)

# Summarise image width and height for each dataset split.
display(
    df_images.groupby("split")[["width", "height"]]
    .agg(["min", "max", "mean"])
)

print("\nIMAGE COLOUR MODES")
print("-" * 40)

# Count RGB, RGBA, grayscale, etc. in each split.
colour_modes = pd.crosstab(
    df_images["split"],
    df_images["mode"]
)

display(colour_modes)

# Identify images that are not RGB.
# These may need converting before RGB feature extraction.
non_rgb_images = df_images[
    df_images["mode"].notna()
    & (df_images["mode"] != "RGB")
].copy()

print(f"\nNon-RGB images: {len(non_rgb_images)}")

if not non_rgb_images.empty:

    display(
        non_rgb_images[
            ["split", "filename", "mode", "path"]
        ]
    )

else:
    print("All readable images use RGB colour mode.")

IMAGE DIMENSIONS
----------------------------------------


width             height            
        min  max   mean    min  max   mean
split                                     
test    720  720  720.0    540  540  540.0
train   720  720  720.0    540  540  540.0
valid   720  720  720.0    540  540  540.0


IMAGE COLOUR MODES
----------------------------------------


mode,RGB
split,
test,80
train,3347
valid,120



Non-RGB images: 0
All readable images use RGB colour mode.


## Find exact duplicates

Group files with identical hashes and count redundant copies beyond one retained file per group.

In [15]:
# ---------------------------------------------------------
# EXACT DUPLICATE IMAGE CHECK
# ---------------------------------------------------------

# Identify all files sharing an identical SHA-256 hash.
# keep=False includes every image in each duplicate group.
duplicate_images = df_images[
    df_images["sha256"].notna()
    & df_images["sha256"].duplicated(keep=False)
].sort_values(["sha256", "split"]).copy()

# Count unique groups of identical files.
num_duplicate_groups = duplicate_images["sha256"].nunique()

# Count redundant copies beyond one file per group.
num_redundant_copies = (
    len(duplicate_images) - num_duplicate_groups
)

print("EXACT DUPLICATE IMAGE CHECK")
print("-" * 40)

print(f"Duplicate groups: {num_duplicate_groups}")
print(f"Images in duplicate groups: {len(duplicate_images)}")
print(f"Redundant copies: {num_redundant_copies}")

if not duplicate_images.empty:

    display(
        duplicate_images[
            ["split", "filename", "path", "sha256"]
        ]
    )

    # Print individual duplicate groups.
    for group_number, (image_hash, group) in enumerate(
        duplicate_images.groupby("sha256"),
        start=1
    ):

        print(f"\nDuplicate group {group_number}:")

        for _, image in group.iterrows():
            print(f"  [{image['split']}] {image['filename']}")

else:
    print("No exact duplicate images detected.")

EXACT DUPLICATE IMAGE CHECK
----------------------------------------
Duplicate groups: 0
Images in duplicate groups: 0
Redundant copies: 0
No exact duplicate images detected.


## Check duplicates across splits

Identify identical files shared by training, validation or test data. Shared files can compromise the independence of evaluation.

In [16]:
# ---------------------------------------------------------
# CROSS-SPLIT DUPLICATE CHECK
# ---------------------------------------------------------

# Count the number of distinct splits containing each image hash.
hash_split_counts = (
    df_images.dropna(subset=["sha256"])
    .groupby("sha256")["split"]
    .nunique()
)

# Select hashes appearing in more than one split.
cross_split_hashes = hash_split_counts[
    hash_split_counts > 1
].index

# Retrieve all images associated with these hashes.
cross_split_duplicates = df_images[
    df_images["sha256"].isin(cross_split_hashes)
].sort_values(["sha256", "split"]).copy()

print("CROSS-SPLIT DUPLICATE CHECK")
print("-" * 40)

print(
    f"Duplicate groups spanning multiple splits: "
    f"{len(cross_split_hashes)}"
)

print(
    f"Images involved in cross-split duplication: "
    f"{len(cross_split_duplicates)}"
)

if not cross_split_duplicates.empty:

    display(
        cross_split_duplicates[
            ["split", "filename", "path", "sha256"]
        ]
    )

    # Display which splits share each duplicate image.
    for group_number, (_, group) in enumerate(
        cross_split_duplicates.groupby("sha256"),
        start=1
    ):

        splits = sorted(group["split"].unique())

        print(
            f"Group {group_number}: "
            f"Appears in {', '.join(splits)}"
        )

else:
    print("No exact duplicates found between dataset splits.")

CROSS-SPLIT DUPLICATE CHECK
----------------------------------------
Duplicate groups spanning multiple splits: 0
Images involved in cross-split duplication: 0
No exact duplicates found between dataset splits.


## Check repeated filenames

Inspect repeated names separately from content hashes: matching filenames alone do not prove identical images.

In [17]:
# ---------------------------------------------------------
# DUPLICATE FILENAME CHECK
# ---------------------------------------------------------

# Identify repeated filenames across all three folders.
duplicate_filenames = df_images[
    df_images["filename"].duplicated(keep=False)
].sort_values(["filename", "split"]).copy()

print("DUPLICATE FILENAME CHECK")
print("-" * 40)

print(f"Files with repeated filenames: {len(duplicate_filenames)}")

if not duplicate_filenames.empty:

    display(
        duplicate_filenames[
            ["split", "filename", "path", "sha256"]
        ]
    )

else:
    print("No repeated filenames detected.")

DUPLICATE FILENAME CHECK
----------------------------------------
Files with repeated filenames: 0
No repeated filenames detected.


## Summarise dataset checks

Review the scan results before considering any cleanup. The summary reports the current in-memory scan.

In [18]:
# ---------------------------------------------------------
# DATASET VALIDATION SUMMARY
# ---------------------------------------------------------

# Count images in each split.
split_counts = df_images["split"].value_counts()

# Calculate validation statistics.
num_images = len(df_images)
num_invalid = len(invalid_images)
num_non_rgb = len(non_rgb_images)
num_duplicate_groups = duplicate_images["sha256"].nunique()
num_cross_split_groups = len(cross_split_hashes)
num_duplicate_filenames = len(duplicate_filenames)

print("=" * 50)
print("       DATASET VALIDATION SUMMARY")
print("=" * 50)

print(f"Total images:                  {num_images}")

print("\nImages by dataset split:")

for split in DATASET_DIRS:
    print(f"  {split:<10} {split_counts.get(split, 0)}")

print("\nValidation results:")

print(f"  Invalid images:             {num_invalid}")
print(f"  Non-RGB images:             {num_non_rgb}")
print(f"  Exact duplicate groups:     {num_duplicate_groups}")
print(f"  Redundant exact copies:     {num_redundant_copies}")
print(f"  Cross-split duplicate groups: {num_cross_split_groups}")
print(f"  Files with repeated names:  {num_duplicate_filenames}")

print("\n" + "-" * 50)

# Determine whether all basic validation checks passed.
# Repeated filenames are informational rather than automatic failures.
if (
    num_invalid == 0
    and num_duplicate_groups == 0
    and num_non_rgb == 0
):

    print("RESULT: Dataset passed basic validation.")

else:

    print("RESULT: Dataset requires review.")

    if num_invalid > 0:
        print("- Review invalid or corrupted images.")

    if num_duplicate_groups > 0:
        print("- Review exact duplicate images.")

    if num_cross_split_groups > 0:
        print("- Resolve cross-split duplicate images.")

    if num_non_rgb > 0:
        print("- Review non-RGB image formats.")

       DATASET VALIDATION SUMMARY
Total images:                  3547

Images by dataset split:
  train      3347
  valid      120
  test       80

Validation results:
  Invalid images:             0
  Non-RGB images:             0
  Exact duplicate groups:     0
  Redundant exact copies:     0
  Cross-split duplicate groups: 0
  Files with repeated names:  0

--------------------------------------------------
RESULT: Dataset passed basic validation.


## Visually inspect duplicate groups

Display each duplicate group to support review of the hash results.

In [19]:
import matplotlib.pyplot as plt

# Display each group of duplicate images for visual inspection.
for group_number, (_, group) in enumerate(
    duplicate_images.groupby("sha256"),
    start=1
):

    fig, axes = plt.subplots(
        1, len(group),
        figsize=(5 * len(group), 5),
        squeeze=False
    )

    for ax, (_, row) in zip(axes[0], group.iterrows()):

        with Image.open(row["path"]) as img:
            ax.imshow(img.convert("RGB"))

        ax.set_title(
            f"{row['split']}: {row['filename']}",
            fontsize=10
        )

        ax.axis("off")

    fig.suptitle(f"Duplicate Group {group_number}")
    plt.tight_layout()
    plt.show()

## Match duplicates to COCO annotations

Look up duplicate filenames in each split’s annotation file and count their recorded regions. This cell assumes duplicate records exist; an empty result can leave the expected DataFrame columns undefined.

In [ ]:
import json

# ---------------------------------------------------------
# CHECK COCO ANNOTATIONS FOR DUPLICATE IMAGES
# ---------------------------------------------------------

# Load COCO annotation data for each dataset split.
coco_data = {}

for split, folder in DATASET_DIRS.items():

    annotation_path = folder / "_annotations.coco.json"

    if not annotation_path.exists():
        print(f"WARNING: No COCO file found for {split}")
        continue

    with open(annotation_path, "r", encoding="utf-8") as file:
        coco_data[split] = json.load(file)

    print(f"Loaded COCO annotations for {split}")


# ---------------------------------------------------------
# MATCH DUPLICATE IMAGES TO THEIR COCO RECORDS
# ---------------------------------------------------------

annotation_records = []

for group_number, (_, group) in enumerate(
    duplicate_images.groupby("sha256"),
    start=1
):

    for _, row in group.iterrows():

        split = row["split"]
        filename = row["filename"]

        # Default results before looking up the image.
        image_found = False
        annotation_count = 0

        if split in coco_data:

            coco = coco_data[split]

            # Find image metadata matching the filename.
            matching_images = [
                image
                for image in coco.get("images", [])
                if Path(image["file_name"]).name == filename
            ]

            image_found = len(matching_images) > 0

            # Retrieve all matching image IDs.
            image_ids = {
                image["id"]
                for image in matching_images
            }

            # Count annotations associated with those image IDs.
            annotation_count = sum(
                annotation["image_id"] in image_ids
                for annotation in coco.get("annotations", [])
            )

        annotation_records.append({
            "duplicate_group": group_number,
            "split": split,
            "filename": filename,
            "coco_image_found": image_found,
            "annotation_count": annotation_count,
            "has_defect_annotations": annotation_count > 0
        })


# ---------------------------------------------------------
# DISPLAY RESULTS
# ---------------------------------------------------------

df_duplicate_annotations = pd.DataFrame(annotation_records)

display(df_duplicate_annotations)

print("\nSUMMARY")
print("-" * 40)

Loaded COCO annotations for train
Loaded COCO annotations for valid
Loaded COCO annotations for test


""



SUMMARY
----------------------------------------


## Compare duplicate annotations

Compare annotation content independently of record ordering. Each duplicate must have a matching COCO record in a loaded annotation file.

In [21]:
# ---------------------------------------------------------
# COMPARE ANNOTATION CONTENTS BETWEEN DUPLICATES
# ---------------------------------------------------------


for group_number, (_, group) in enumerate(
    duplicate_images.groupby("sha256"),
    start=1
):
    annotation_sets = []

    for _, row in group.iterrows():

        coco = coco_data[row["split"]]

        # Find the matching COCO image record.
        image_record = next(
            img for img in coco["images"]
            if Path(img["file_name"]).name == row["filename"]
        )

        # Retrieve annotations belonging to this image.
        annotations = [
            {
                "category_id": ann.get("category_id"),
                "bbox": ann.get("bbox"),
                "segmentation": ann.get("segmentation"),
                "area": ann.get("area"),
                "iscrowd": ann.get("iscrowd", 0)
            }
            for ann in coco["annotations"]
            if ann["image_id"] == image_record["id"]
        ]

        # Sort annotations to make the comparison independent
        # of their order in the COCO JSON file.
        annotation_sets.append(
            sorted(json.dumps(ann, sort_keys=True) for ann in annotations)
        )

    # Check whether all copies have identical annotation contents.
    are_identical = all(
        annotations == annotation_sets[0]
        for annotations in annotation_sets[1:]
    )

    print(
        f"Duplicate Group {group_number}: "
        f"{'IDENTICAL ANNOTATIONS' if are_identical else 'DIFFERENT ANNOTATIONS'}"
    )

## Existing cleanup step: modifies training data

**This cell changes files when run.** It backs up the training annotation file, moves redundant training images to the backup folder and updates training COCO records. Review the duplicate and annotation checks first. It does not resolve all possible cross-split duplicates. The readability review has not executed this cell.

In [22]:
import shutil

# ---------------------------------------------------------
# REMOVE DUPLICATE TRAINING IMAGES AND COCO ANNOTATIONS
# ---------------------------------------------------------

TRAIN_DIR = DATASET_DIRS["train"]
COCO_PATH = TRAIN_DIR / "_annotations.coco.json"

# Store backups outside the raw dataset directories.
BACKUP_DIR = TRAIN_DIR.parent.parent / "duplicate_backups"
BACKUP_DIR.mkdir(parents=True, exist_ok=True)

# Load the original COCO annotation file.
with open(COCO_PATH, "r", encoding="utf-8") as file:
    coco = json.load(file)

# Create a backup of the COCO file before making changes.
backup_coco_path = BACKUP_DIR / "_annotations.coco.original.json"

if not backup_coco_path.exists():
    shutil.copy2(COCO_PATH, backup_coco_path)

# Select redundant copies from each duplicate group.
# The first copy is retained; all subsequent copies are removed.
images_to_remove = []

train_duplicates = duplicate_images[
    duplicate_images["split"] == "train"
]

for _, group in train_duplicates.groupby("sha256"):

    # Keep the first file in the group.
    redundant_images = group.iloc[1:]

    for _, row in redundant_images.iterrows():
        images_to_remove.append(Path(row["path"]))

# Find COCO image IDs belonging to redundant files.
filenames_to_remove = {
    path.name for path in images_to_remove
}

image_ids_to_remove = {
    image["id"]
    for image in coco["images"]
    if Path(image["file_name"]).name in filenames_to_remove
}

# Verify that every selected image has a COCO record.
if len(image_ids_to_remove) != len(images_to_remove):
    raise ValueError(
        "Mismatch between duplicate files and COCO image records. "
        "No changes have been made."
    )

print("Removing redundant copies:")
for path in images_to_remove:
    print(f"  {path.name}")

# Remove selected image entries from COCO.
coco["images"] = [
    image for image in coco["images"]
    if image["id"] not in image_ids_to_remove
]

# Remove annotations associated with deleted image IDs.
coco["annotations"] = [
    ann for ann in coco["annotations"]
    if ann["image_id"] not in image_ids_to_remove
]

# Move redundant images into the backup directory.
for image_path in images_to_remove:

    destination = BACKUP_DIR / image_path.name

    if destination.exists():
        raise FileExistsError(
            f"Backup already exists: {destination}"
        )

    shutil.move(str(image_path), str(destination))

# Save the updated COCO annotation file.
with open(COCO_PATH, "w", encoding="utf-8") as file:
    json.dump(coco, file, indent=2)

print("\nDuplicate cleanup complete.")
print(f"Images removed: {len(images_to_remove)}")
print(f"COCO image records removed: {len(image_ids_to_remove)}")
print("Original COCO file and removed images saved in:")
print(BACKUP_DIR)

Removing redundant copies:



Duplicate cleanup complete.
Images removed: 0
COCO image records removed: 0
Original COCO file and removed images saved in:
..\data\duplicate_backups
